# Strands Agent with Dash0 Observability on Amazon Bedrock AgentCore Runtime

## Overview

This notebook demonstrates deploying a Strands agent to Amazon Bedrock AgentCore Runtime with Dash0 integration. The implementation uses Amazon Bedrock Claude models and sends telemetry data to Dash0 through OpenTelemetry (OTEL).

## Key Components

- **Strands Agents**: Python framework for building LLM-powered agents with built-in telemetry support
- **Amazon Bedrock AgentCore Runtime**: Managed runtime service for hosting and scaling agents on AWS
- **Dash0**: OpenTelemetry-native observability platform for monitoring distributed systems and AI agents
- **OpenTelemetry**: Industry-standard protocol for collecting and exporting telemetry data

## Architecture

The agent is containerized and deployed to AgentCore Runtime, which provides HTTP endpoints for invocation. Telemetry data flows from the Strands agent through an OTLP exporter directly to Dash0's ingress endpoint for monitoring and debugging. The implementation disables AgentCore's default ADOT observability to use Dash0 instead.

## Prerequisites

- Python 3.10+
- AWS credentials configured with Bedrock and AgentCore permissions
- [Dash0](https://www.dash0.com/) account with an auth token
- Docker installed locally
- Access to Amazon Bedrock Claude models in your configured region

In [ ]:
!pip install --force-reinstall -U -r requirements.txt

## Configure Credentials

Create a `.env` file in this directory with your API keys:

```
AWS_DEFAULT_REGION=us-east-1
DASH0_AUTH_TOKEN=your-dash0-auth-token
DASH0_OTLP_ENDPOINT=https://ingress.us-west-2.aws.dash0.com/v1/traces
DASH0_DATASET=default
```

You can obtain your Dash0 auth token from **Settings → Auth Tokens** in the [Dash0 UI](https://app.dash0.com/).

The OTLP ingress endpoint depends on the region of your Dash0 organization:
- **US West 2**: `https://ingress.us-west-2.aws.dash0.com/v1/traces`
- **EU West 1**: `https://ingress.eu-west-1.aws.dash0.com/v1/traces`

Find your specific endpoint in the Dash0 UI under **Settings → Endpoints**.

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
# Safe verification (no secrets printed)
import boto3
try:
    resp = boto3.client('sts').get_caller_identity()
    print('AWS identity:', {k: resp[k] for k in ('Account','Arn','UserId') if k in resp})
except Exception as e:
    print('Credential check failed:', type(e).__name__, str(e))

## Agent Implementation

The agent file (`strands_claude.py`) implements a travel assistant with calculator and weather tools. Key configuration includes:
- **`DISABLE_ADOT_OBSERVABILITY=true`**: Disables AgentCore's built-in ADOT pipeline so we can set our own TracerProvider ([AgentCore observability docs](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-configure.html))
- **Direct OTLP export**: Configures an OpenTelemetry TracerProvider that exports traces to the Dash0 OTLP ingress endpoint
- **Auth header**: Set the OTLP exporter `Authorization: Bearer <DASH0_AUTH_TOKEN>` header for authentication
- **Dataset routing**: The optional `Dash0-Dataset` header routes traces to a named dataset within your Dash0 organization
- **`OTEL_SEMCONV_STABILITY_OPT_IN=gen_ai_latest_experimental`**: Enables OpenTelemetry v1.37+ GenAI semantic conventions required by Strands Agents ([Strands observability docs](https://strandsagents.com/latest/documentation/docs/user-guide/observability-evaluation/observability/))
- **Automatic trace export**: All agent invocations, tool calls, and LLM interactions are automatically traced and sent to Dash0 once the exporter is configured

In [ ]:
%%writefile strands_claude.py
import os
import logging

logging.basicConfig(level=logging.ERROR, format="[%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(os.getenv("AGENT_RUNTIME_LOG_LEVEL", "INFO").upper())

# =============================================================================
# Dash0 - OpenTelemetry Configuration
# Must be configured BEFORE any other OpenTelemetry imports
# =============================================================================

# Disable AgentCore's built-in ADOT so we can set our own TracerProvider
# See: https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-configure.html
os.environ["DISABLE_ADOT_OBSERVABILITY"] = "true"

# Required for strands-agents GenAI semantic conventions (v1.37+)
os.environ["OTEL_SEMCONV_STABILITY_OPT_IN"] = "gen_ai_latest_experimental"

dash0_auth_token = os.environ.get("DASH0_AUTH_TOKEN")
dash0_endpoint = os.environ.get("DASH0_OTLP_ENDPOINT", "https://ingress.us-west-2.aws.dash0.com/v1/traces")
dash0_dataset = os.environ.get("DASH0_DATASET", "default")
service_name = os.environ.get("OTEL_SERVICE_NAME", "agentcore-dash0-demo")

if dash0_auth_token:
    from opentelemetry import trace
    from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
    from opentelemetry.sdk.trace import TracerProvider
    from opentelemetry.sdk.trace.export import SimpleSpanProcessor
    from opentelemetry.sdk.resources import Resource

    resource = Resource.create({"service.name": service_name})
    headers = {"Authorization": f"Bearer {dash0_auth_token}"}
    if dash0_dataset:
        headers["Dash0-Dataset"] = dash0_dataset
    exporter = OTLPSpanExporter(
        endpoint=dash0_endpoint,
        headers=headers,
    )
    provider = TracerProvider(resource=resource)
    provider.add_span_processor(SimpleSpanProcessor(exporter))
    trace.set_tracer_provider(provider)
    logger.info("Dash0 observability configured (service: %s, dataset: %s)", service_name, dash0_dataset)
else:
    logger.warning("DASH0_AUTH_TOKEN not set. Traces will not be sent to Dash0.")

# =============================================================================
# Agent
# =============================================================================

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import calculator


def get_bedrock_model():
    region = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
    model_id = os.getenv("BEDROCK_MODEL_ID", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
    return BedrockModel(
        model_id=model_id,
        region_name=region,
        max_tokens=1024,
    )


bedrock_model = get_bedrock_model()

system_prompt = """You are a helpful travel assistant. You can perform mathematical calculations 
and check weather information. Always provide helpful, accurate responses and use tools when appropriate."""


@tool
def weather():
    """Get current weather."""
    return "sunny and 72F"


app = BedrockAgentCoreApp()


def initialize_agent():
    """Initialize the agent (telemetry is already configured at module level)."""
    return Agent(
        model=bedrock_model,
        system_prompt=system_prompt,
        tools=[calculator, weather],
    )


@app.entrypoint
def strands_agent_bedrock(payload, context=None):
    """Invoke the agent with a payload."""
    user_input = payload.get("prompt", payload.get("text", payload.get("message", "Hello")))
    logger.info("[%s] User input: %s", getattr(context, 'session_id', 'local'), user_input)

    agent = initialize_agent()
    response = agent(user_input)
    return response.message['content'][0]['text']


if __name__ == "__main__":
    app.run()

### Configure AgentCore Runtime deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code. Please note that when using the `bedrock_agentcore_starter_toolkit` to configure your agent, it configures AgentCore Observability by default so, to use Dash0, you need to remove configuration for AgentCore Observability as explained below:

<div style="text-align:left">
    <img src="../images/configure.png" width="40%"/>
</div>

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "strands_dash0_agent"
response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    memory_mode="NO_MEMORY",
    disable_otel=True,
)
response

## Deploy to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime.

### Dash0 Configuration

To send traces to Dash0, you need:
- **Dash0 Auth Token**: Get this from your Dash0 account at **Settings → Auth Tokens** in the [Dash0 UI](https://app.dash0.com/)
- **OTLP Endpoint**: The ingress URL for your Dash0 organization's region

The agent code (`strands_claude.py`) automatically configures all OTLP settings when `DASH0_AUTH_TOKEN` is provided:
- **Endpoint**: `https://ingress.us-west-2.aws.dash0.com/v1/traces` (override with `DASH0_OTLP_ENDPOINT`)
- **Auth header**: `Authorization: Bearer {token}`
- **Dataset header**: `Dash0-Dataset: {dataset}` — routes traces to a named dataset (override with `DASH0_DATASET`)
- **Semantic Conventions**: `gen_ai_latest_experimental` (for GenAI-aware trace views)

**For other Dash0 regions**, set `DASH0_OTLP_ENDPOINT` in the launch env_vars:
- US West 2 (AWS): `https://ingress.us-west-2.aws.dash0.com/v1/traces` (default)
- EU West 1 (AWS): `https://ingress.eu-west-1.aws.dash0.com/v1/traces`

Find your specific endpoint under **Settings → Endpoints** in the Dash0 UI.

<div style="text-align:left">
    <img src="../images/launch.png" width="75%"/>
</div>

In [ ]:
%load_ext dotenv
%dotenv
import os

# Dash0 configuration
dash0_auth_token = os.environ.get("DASH0_AUTH_TOKEN")  # Replace with your Dash0 auth token

launch_result = agentcore_runtime.launch(
    env_vars={
        "DASH0_AUTH_TOKEN": dash0_auth_token,
        "DASH0_OTLP_ENDPOINT": "https://ingress.us-west-2.aws.dash0.com/v1/traces",  # Change for other regions
        "DASH0_DATASET": "default",  # Dataset to store traces in
        "OTEL_SERVICE_NAME": "agentcore-dash0-demo",
        "DISABLE_ADOT_OBSERVABILITY": "true",  # Disable AgentCore's default observability
    },
)
launch_result

## Check Deployment Status

Wait for the runtime to be ready before invoking:

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

<div style="text-align:left">
    <img src="../images/invoke.png" width="75%"/>
</div>

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "What is 25 + 17 and what's the weather like for a trip to Paris?"})

In [ ]:
from IPython.display import Markdown, display
display(Markdown("".join(invoke_response['response'])))

## View Traces in Dash0

After invoking the agent, traces appear in Dash0 within seconds:

1. Go to [Dash0](https://app.dash0.com/)
2. Navigate to **Tracing** in the left sidebar
3. Filter by `service.name = agentcore-dash0-demo` or browse the `default` dataset

The traces will include:
- Agent invocation details with full request/response context
- Tool calls (calculator, weather) with execution time
- Model interactions with latency and token usage
- GenAI semantic convention attributes (model name, input/output token counts, prompt/completion content via span events)

### Dash0 Observability Features

Dash0 is built natively on OpenTelemetry, providing purpose-built views for GenAI and distributed applications:

- **Trace Explorer**: View end-to-end agent traces with full span details, attributes, and events in a timeline view
- **GenAI Attributes**: First-class support for OpenTelemetry GenAI semantic conventions — model names, token counts, prompt/completion content surface automatically
- **Datasets**: Organize traces by dataset for multi-service or multi-environment setups
- **Metrics & Logs**: Correlate traces with metrics and logs in a single pane using native OTLP ingestion
- **Alerting**: Set up alerts on trace-derived metrics such as error rates and latency thresholds
- **No Vendor Lock-in**: Dash0 stores and queries data in open formats — export to any backend at any time

For more information, see the [Dash0 documentation](https://dash0.com/docs).

## Cleanup (Optional)

Clean up the deployed resources:

In [ ]:
import boto3

agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)

ecr_client = boto3.client(
    'ecr',
    region_name=region
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

## Summary

You have successfully deployed a Strands agent to Amazon Bedrock AgentCore Runtime with Dash0 observability. The implementation demonstrates:
- Disabling AgentCore's built-in ADOT to use a custom observability provider
- Configuring an OpenTelemetry TracerProvider to export traces directly to Dash0 via OTLP HTTP
- Using `Authorization: Bearer` and `Dash0-Dataset` headers for authentication and dataset routing
- Enabling GenAI semantic conventions for LLM-specific trace attributes
- Invocation through the AgentCore starter toolkit SDK

### Resources

- [AgentCore Observability docs](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/observability-configure.html)
- [Dash0 Documentation](https://dash0.com/docs)
- [Dash0 OpenTelemetry Ingestion](https://dash0.com/docs/dash0/miscellaneous/glossary/endpoints)
- [Strands Agents Observability](https://strandsagents.com/latest/documentation/docs/user-guide/observability-evaluation/observability/)
- [OpenTelemetry GenAI Semantic Conventions](https://opentelemetry.io/docs/specs/semconv/gen-ai/)